# Category Composition Pie Chart

This notebook generates and exports a pie chart showing the composition of a category at a location. To use it, update the location, category and required export format in the first code cell, below, before running the notebook.

In [ ]:
# Location to report on
location = "Abingdon"

# Category to report on
category = "Birds"

# Export format for the trend chart:
# PNG     - export as PNG image
# PDF     - export as PDF file
# <blank> - do not export
export_format = "PNG"

In [ ]:
%run pathutils.ipynb
%run database.ipynb
%run export.ipynb

In [ ]:
import sqlparse

# Construct the query
query = construct_query("category_composition.sql", {
    "LOCATION": location,
    "CATEGORY": category
})

# Show a pretty-printed form of the query
print(sqlparse.format(query, reindent=True, keyword_case='upper'))

In [ ]:
# Connect to the database and execute the query and read the results into a dataframe
df = query_data(query)

# Check there is some data
if not df.shape[0]:
    message = f"No data found for category '{category}' at location '{location}'"
    raise ValueError(message)

In [ ]:
import pandas as pd
import re

# Group by species and sum number of individuals
species_counts = df.groupby("Species")["Count"].sum()

# Sort values (optional, for nicer plots)
species_counts = species_counts.sort_values(ascending=False)

# Create the folder to hold exported reports
export_folder_path = get_export_folder_path()

# Export the data to Excel
clean_location = clean_string(location)
clean_category = clean_string(category)
export_file_name = f"{clean_category}-{clean_location}-Category-Composition"
export_to_spreadsheet(export_folder_path, f"{export_file_name}.xlsx", {
    "Species Counts": species_counts
})

# Print the scatter plot data
with pd.option_context('display.max_rows', None,
                       'display.max_columns', None,
                       'display.precision', 3,
                       ):
    display(species_counts)

In [ ]:

import matplotlib.pyplot as plt

# Prepare data
labels = species_counts.index
sizes = species_counts.values
percentages = sizes / sizes.sum() * 100
legend_labels = [f"{label} ({pct:.1f}%)" for label, pct in zip(labels, percentages)]

# Create figure with extra vertical space for legend
fig, ax = plt.subplots(figsize=(8, 8))

wedges, _ = ax.pie(sizes, startangle=90, counterclock=False)

# Equal aspect ratio
ax.set_aspect('equal')
ax.set_title(f"Composition of '{category}' Category by Species")

# Remove tight_layout, manually control layout instead
fig.subplots_adjust(bottom=0.3)  # Increase this if needed

# Place legend fully outside the plot
ax.legend(wedges, legend_labels,
            title="Species",
            loc='upper center',
            bbox_to_anchor=(0.5, -0.005),
            ncol=3,
            frameon=False)

# Export the chart
export_chart(export_folder_path, export_file_name, export_format)

# Show the plot
plt.show()